# ఖర్చు క్లెయిమ్ విశ్లేషణ

ఈ నోట్‌బుక్ లోకల్ రసీదు చిత్రాల నుండి ప్రయాణ ఖర్చులను ప్రాసెస్ చేయడానికి ప్లగిన్లను ఉపయోగించే ఏజెంట్లను ఎలా సృష్టించాలో చూపిస్తుంది, ఖర్చు క్లెయిమ్ ఇమెయిల్ను ఉత్పత్తి చేస్తుంది, మరియు పై చార్ట్ ఉపయోగించి ఖర్చుల డేటాను విజువలైజ్ చేస్తుంది. ఏజెంట్లు టాస్క్ సందర్భాన్ని ఆధారంగా ఫంక్షన్లను డైనమిక్‌గా ఎంచుకుంటాయి.

దశలు:
1. OCR ఏజెంట్ లోకల్ రసీదు చిత్రాన్ని ప్రాసెస్ చేస్తుంది మరియు ప్రయాణ ఖర్చు డేటాను ఎక్స్‌ట్రాక్ట్ చేస్తుంది.
2. ఇమెయిల్ ఏజెంట్ ఖర్చు క్లెయిమ్ ఇమెయిల్ను ఉత్పత్తి చేస్తుంది.

### ప్రయాణ ఖర్చుల సన్నివేశానికి ఉదాహరణ:
మీరు మరో పట్టణంలో వ్యాపార సమావేశానికి ప్రయాణిస్తున్న ఉద్యోగి అని ఊహించుకోండి. మీ సంస్థ అన్ని సమంజసమైన ప్రయాణ సంబంధిత ఖర్చులను తిరిగి చెల్లించే విధానాన్ని కలిగి ఉంది. ఇక్కడ సాధ్యమైన ప్రయాణ ఖర్చులు వివరించబడ్డాయి:
- రవాణా:
మీ గృహ పట్టణం నుండి గమ్యస్థాన పట్టణానికి రౌండ్ ట్రిప్ కోసం ఎయిర్‌ఫేర్.
ఎయిర్‌పోర్టుకు మరియు తిరిగి వెళ్లడానికి టాక్సీ లేదా రైడ్-హెయిలింగ్ సేవలు.
గమ్యస్థాన పట్టణంలో లోకల్ రవాణా (పబ్లిక్ ట్రాన్సిట్, రెంటల్ కార్లు, లేదా టాక్సీలు వంటి).

- నివాసం:
మీటింగ్ వేదికకి సమీపంలోని మధ్యస్థాయి వ్యాపార హోటల్‌లో మూడు రాత్రుల హోటల్ నిలియడం.

- భోజనాలు:
సంస్థ యొక్క పర్ డియమ్ విధానం ఆధారంగా ప్రతి రోజు అలవెన్స్ (ఉదయం, మద్యాహ్నం, రాత్రి భోజనం).

- వివిధ ఖర్చులు:
ఎయిర్‌పోర్ట్ వద్ద పార్కింగ్ ఫీజులు.
హోటల్‌లో ఇంటర్నెట్ యాక్సెస్ చార్జీలు.
టిప్స్ లేదా చిన్న సేవా చార్జీలు.

- డాక్యుమెంటేషన్:
మీరు అన్ని రసీదులను (ఫ్లైట్లు, టాక్సీలు, హోటల్, భోజనాలు, మొదలైనవి) మరియు ఖర్చుల నివేదికను తిరిగి చెల్లింపుకు సమర్పిస్తారు.


## అవసరమైన లైబ్రరీలను దిగుమతి చేసుకోండి

నోట్‌బుక్ కోసం అవసరమైన లైబ్రరీలు మరియు మోడ్యూల్స్‌ను దిగుమతి చేసుకోండి.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## ఖర్చు నమూనాలను నిర్వచించండి

 వ్యక్తిగత ఖర్చులకు Pydantic మోడల్ ను సృష్టించండి మరియు వినియోగదారుల ప్రశ్నను నిర్మితమైన ఖర్చు డేటాగా మార్చడానికి ExpenseFormatter తరగతిని సృష్టించండి.

 ప్రతి ఖర్చు ఈ ఫార్మాట్లో ప్రదర్శించబడుతుంది:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## సాధనాలను నిర్వచించడం - ఇమెయిల్ రూపొందించడం

ఖర్చుల క్లెయిమ్ సమర్పించడానికి ఇమెయిల్ రూపొందించడానికి ఒక సాధన ఫంక్షన్ రూపొందించండి.
- ఈ సాధనం Microsoft Agent Framework నుండి `@tool` డెకరేటర్ ఉపయోగిస్తుంది.
- ఖర్చుల మొత్తం మొత్తాన్ని లెక్కించి వివరాలను ఇమెయిల్ బాడీకి ఫార్మాట్ చేస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# రసీదు చిత్రాల నుండి ట్రావెల్ ఖర్చులను తీయడానికి సాధనం

రసీదు చిత్రాల నుండి ట్రావెల్ ఖర్చులను తీయడానికి ఒక సాధన ఫంక్షన్ సృష్టించండి.
- ఈ సాధనం Microsoft ఏజెంట్ ఫ్రేమ్‌వర్క్ నుండి `@tool` డెకరేటర్ ను ఉపయోగిస్తుంది.
- ఇది రసీదు చిత్రాన్ని చదువుతుంది, దాన్ని base64గా ఎంకోడ్ చేసి, ఏజెంట్ విశ్లేషించడానికి డేటా URIని తిరిగి ఇస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## ఖర్చుల ప్రాసెసింగ్

ఏజెంట్లను నిర్వచించి వాటిని `WorkflowBuilder` ఉపయోగించి అనుక్రమిక వర్క్‌ఫ్లోలో కనెక్ట్ చేయండి.
- OCR ఏజెంట్ రసీదు చిత్రంలోని నిర్మాణాత్మక ఖర్చుల డేటాను `load_receipt_image` టూల్ ఉపయోగించి వెలికి తీయుతుంది.
- ఇమెయిల్ ఏజెంట్ వెలికి తీయబడిన డేటాను తీసుకుని `generate_expense_email` టూల్ ఉపయోగించి వృత్తిపరమైన ఖర్చు క్లెయిమ్ ఇమెయిల్ రూపొందిస్తుంది.
- `add_edge` తో `WorkflowBuilder` అనుక్రమిక పైప్‌లైన్‌ను సృష్టిస్తుంది: OCR ఏజెంట్ → ఇమెయిల్ ఏజెంట్.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Main function

ఆనువంశిక వర్క్‌ఫ్లోను నిర్మించి, దాన్ని నడిపించి రశీదుల చిత్రం ప్రాసెస్ చేయండి మరియు ఖర్చుల క్లెయిమ్ ఇమెయిల్‌ను రూపొందించండి.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**హితాబద్ధత**:
ఈ పత్రాన్ని AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మనం సరైనత కోసం ప్రయత్నించినప్పటికీ, ఆటోమేటెడ్ అనువాదాలలో తప్పులు లేదా అసమగ్రతలు ఉండవచ్చు అని దయచేసి గమనించండి. మౌలిక భాషలో ఉన్న అసలు పత్రం అధికారిక మూలంగా పరిగణించబడాలి. కీలక సమాచారానికి నిపుణుల మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదాన్ని ఉపయోగించడంనుండి ఏర్పడిన ఏ ఇతరార్థాలు లేదా తప్పు అర్థాలకు మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
